# Archie 3D — Blender headless explainer (In Plain Motion style)
Topic: Why ice is slippery — short vertical 3D science short for YouTube Shorts.

In [ ]:
import os, sys, subprocess, pathlib, json, shutil, time
OUT = pathlib.Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
FRAMES = OUT / 'frames'
FRAMES.mkdir(exist_ok=True)
print('cwd', os.getcwd())
print('gpu check')
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Install Blender official Linux binary (headless)
BLENDER_VER = '4.2.3'
BLENDER_URL = f'https://download.blender.org/release/Blender4.2/blender-{BLENDER_VER}-linux-x64.tar.xz'
BLENDER_DIR = pathlib.Path('/kaggle/working/blender')
BLENDER_BIN = BLENDER_DIR / f'blender-{BLENDER_VER}-linux-x64' / 'blender'
if not BLENDER_BIN.exists():
    print('Downloading Blender', BLENDER_VER)
    tar = OUT / 'blender.tar.xz'
    subprocess.check_call(['curl', '-L', '-o', str(tar), BLENDER_URL])
    BLENDER_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['tar', '-xf', str(tar), '-C', str(BLENDER_DIR)])
    tar.unlink(missing_ok=True)
print('blender', BLENDER_BIN, 'exists', BLENDER_BIN.exists())
subprocess.run([str(BLENDER_BIN), '--version'], check=False)


In [ ]:
# Write Blender Python scene: ice surface + footstep concept (In Plain Motion style)
scene_py = OUT / 'build_ice_scene.py'
scene_py.write_text('''
import bpy
import math
from mathutils import Vector

# Reset
bpy.ops.wm.read_factory_settings(use_empty=True)
scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
try:
    prefs = bpy.context.preferences.addons['cycles'].preferences
    prefs.compute_device_type = 'CUDA'
    prefs.get_devices()
    for d in prefs.devices:
        d.use = True
except Exception as e:
    print('GPU setup soft-fail', e)
    scene.cycles.device = 'CPU'

scene.cycles.samples = 24
scene.cycles.use_denoising = True
scene.render.resolution_x = 1080
scene.render.resolution_y = 1920
scene.render.resolution_percentage = 100
scene.render.fps = 24
scene.frame_start = 1
scene.frame_end = 120  # 5 seconds
scene.render.image_settings.file_format = 'PNG'
scene.render.filepath = '/kaggle/working/frames/frame_'

# World
world = bpy.data.worlds.new('World')
scene.world = world
world.use_nodes = True
bg = world.node_tree.nodes['Background']
bg.inputs[0].default_value = (0.55, 0.72, 0.88, 1)
bg.inputs[1].default_value = 0.8

# Ice plane
bpy.ops.mesh.primitive_plane_add(size=8, location=(0, 0, 0))
ice = bpy.context.active_object
ice.name = 'Ice'
mat_ice = bpy.data.materials.new('IceMat')
mat_ice.use_nodes = True
nt = mat_ice.node_tree
nodes, links = nt.nodes, nt.links
nodes.clear()
out = nodes.new('ShaderNodeOutputMaterial')
glass = nodes.new('ShaderNodeBsdfGlass')
glass.inputs['Color'].default_value = (0.75, 0.9, 1.0, 1)
glass.inputs['Roughness'].default_value = 0.08
glass.inputs['IOR'].default_value = 1.31
links.new(glass.outputs[0], out.inputs[0])
ice.data.materials.append(mat_ice)

# Subtle ice cracks as thin cubes
for i, (x, y) in enumerate([(-1.2, 0.4), (0.8, -0.6), (0.1, 1.1), (-0.5, -1.0)]):
    bpy.ops.mesh.primitive_cube_add(size=1, location=(x, y, 0.01))
    crack = bpy.context.active_object
    crack.scale = (1.4, 0.02, 0.005)
    crack.rotation_euler[2] = 0.4 * i
    mat_c = bpy.data.materials.new(f'Crack{i}')
    mat_c.use_nodes = True
    mat_c.node_tree.nodes['Principled BSDF'].inputs['Base Color'].default_value = (0.9, 0.95, 1, 1)
    mat_c.node_tree.nodes['Principled BSDF'].inputs['Roughness'].default_value = 0.3
    crack.data.materials.append(mat_c)

# Boot / foot simplified
bpy.ops.mesh.primitive_cube_add(size=1, location=(0, -2.5, 0.35))
boot = bpy.context.active_object
boot.name = 'Boot'
boot.scale = (0.45, 0.9, 0.25)
mat_b = bpy.data.materials.new('BootMat')
mat_b.use_nodes = True
mat_b.node_tree.nodes['Principled BSDF'].inputs['Base Color'].default_value = (0.15, 0.12, 0.1, 1)
mat_b.node_tree.nodes['Principled BSDF'].inputs['Roughness'].default_value = 0.7
boot.data.materials.append(mat_b)

# Animate boot sliding forward (why ice is slippery — low friction)
boot.location = (0, -2.2, 0.35)
boot.keyframe_insert(data_path='location', frame=1)
boot.location = (0.15, 1.8, 0.35)
boot.keyframe_insert(data_path='location', frame=90)
boot.location = (0.25, 2.4, 0.55)
boot.keyframe_insert(data_path='location', frame=120)
# slight tilt
boot.rotation_euler = (0.05, 0, 0.02)
boot.keyframe_insert(data_path='rotation_euler', frame=1)
boot.rotation_euler = (0.12, 0.05, -0.08)
boot.keyframe_insert(data_path='rotation_euler', frame=100)

# Camera vertical short
cam_data = bpy.data.cameras.new('Cam')
cam = bpy.data.objects.new('Cam', cam_data)
scene.collection.objects.link(cam)
scene.camera = cam
cam.location = (3.2, -3.5, 2.8)
cam.rotation_euler = (math.radians(58), 0, math.radians(42))
cam_data.lens = 35

# Lights
bpy.ops.object.light_add(type='SUN', location=(2, -2, 6))
sun = bpy.context.active_object
sun.data.energy = 3.5
sun.rotation_euler = (math.radians(40), math.radians(15), math.radians(20))
bpy.ops.object.light_add(type='AREA', location=(-2, 1, 3))
area = bpy.context.active_object
area.data.energy = 40
area.data.size = 4

# Water film thin plane above ice (melt layer idea)
bpy.ops.mesh.primitive_plane_add(size=7.5, location=(0, 0, 0.02))
film = bpy.context.active_object
film.name = 'WaterFilm'
mat_w = bpy.data.materials.new('WaterFilm')
mat_w.use_nodes = True
wn = mat_w.node_tree.nodes
wn.clear()
wo = wn.new('ShaderNodeOutputMaterial')
wg = wn.new('ShaderNodeBsdfGlass')
wg.inputs['Color'].default_value = (0.85, 0.95, 1, 1)
wg.inputs['Roughness'].default_value = 0.02
wg.inputs['IOR'].default_value = 1.33
mat_w.node_tree.links.new(wg.outputs[0], wo.inputs[0])
film.data.materials.append(mat_w)

print('Scene ready, rendering...')
bpy.ops.render.render(animation=True)
print('Render complete')
''')
print('Wrote', scene_py)


In [ ]:
# Run Blender headless
cmd = [str(BLENDER_BIN), '--background', '--factory-startup', '--python', str(scene_py)]
print('RUN', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print('exit', r.returncode)
print(r.stdout[-2000:] if r.stdout else '')
print(r.stderr[-2000:] if r.stderr else '')
frames = sorted(FRAMES.glob('frame_*.png'))
print('frames', len(frames))


In [ ]:
# Stitch to vertical MP4 + captions overlay metadata
mp4 = OUT / 'archie_ice_slippery_3d.mp4'
if list(FRAMES.glob('frame_*.png')):
    subprocess.check_call([
        'ffmpeg', '-y', '-framerate', '24',
        '-i', str(FRAMES / 'frame_%04d.png'),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
        '-crf', '18', '-movflags', '+faststart',
        str(mp4)
    ])
    print('MP4', mp4, mp4.stat().st_size)
else:
    print('No frames — writing placeholder note')
    (OUT / 'NO_FRAMES.txt').write_text('render produced no frames')

meta = {
    'title': 'Why Ice Is Actually Slippery',
    'niche': 'archie_science_3d',
    'style': 'in_plain_motion',
    'duration_target_sec': 5,
    'script': (
        'Ice is not just smooth. Under your shoe, pressure and friction melt a razor-thin water film. '
        'That liquid layer is what makes you slide. Aim for the science, not the scare.'
    ),
    'hashtags': ['#ArchieExplains', '#ScienceShorts', '#WhyIceIsSlippery', '#3D', '#Blender', '#FYP'],
    'frames': len(list(FRAMES.glob('frame_*.png'))),
    'mp4': str(mp4.name) if mp4.exists() else None,
}
(OUT / 'manifest.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
print('DONE')
